# 🔧 Notebook 02: 数据清洗深入

## 学习目标
1. 理解电力数据中常见的脏数据问题
2. 掌握缺失值处理的前向填充法
3. 理解 IQR 异常值检测的原理
4. 理解为什么电力尖峰**不应该**删除

## 背景：GIGO 原则

> **Garbage In, Garbage Out**（垃圾进，垃圾出）

如果输入模型的数据是脏的，无论模型多复杂，输出一定是脏的。
数据清洗是机器学习管道中最枯燥但**最重要**的环节。

### 电力数据常见的脏数据

| 问题 | 原因 | 例子 |
|------|------|------|
| 缺失值 | 传感器故障、网络中断 | 某小时没有读数 |
| 异常值 | 传感器故障、极端事件 | 负荷突然跳到 10× 正常值 |
| 时区混乱 | 多源数据、夏令时 | UTC 和北京时间混用 |
| 重复值 | 数据采集bug | 同一小时两条记录 |
| 粒度不一致 | 数据源切换 | 小时级和日级数据混在一起 |

In [ ]:
import sys
sys.path.insert(0, '..')
from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data, validate_schema
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

---
## 步骤 1: 加载原始数据

In [ ]:
loader = create_loader('owid')
df_raw = loader.load_data()

print(f"原始数据: {df_raw.shape[0]} 行")
print(f"缺失值统计:")
print(df_raw.isna().sum())

---
## 步骤 2: IQR 异常值检测

### IQR (Interquartile Range, 四分位距) 方法

IQR 是最经典的异常值检测方法之一，由统计学家 John Tukey 在 1977 年提出。

**计算步骤:**
1. 将所有数据从小到大排序
2. 找到 Q1 (第 25 百分位, 即下四分位数)
3. 找到 Q3 (第 75 百分位, 即上四分位数)
4. IQR = Q3 - Q1
5. 下限 = Q1 - 1.5 × IQR
6. 上限 = Q3 + 1.5 × IQR
7. 超出 [下限, 上限] 范围的 → 标记为异常值

**为什么是 1.5？**
这是 Tukey 的经验常数。对于正态分布，约 0.7% 的数据落在 1.5×IQR 之外。
如果取 3×IQR，只有约 0.00006% 的数据被标为异常（过于保守）。

In [ ]:
q1 = df_raw['load_mw'].quantile(0.25)
q3 = df_raw['load_mw'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1 (25%): {q1:,.0f} MW")
print(f"Q3 (75%): {q3:,.0f} MW")
print(f"IQR:     {iqr:,.0f} MW")
print(f"正常范围: [{lower:,.0f}, {upper:,.0f}] MW")

outliers = df_raw[(df_raw['load_mw'] < lower) | (df_raw['load_mw'] > upper)]
print(f"\n统计异常值: {len(outliers)} 个 ({len(outliers)/len(df_raw)*100:.1f}%)")
if len(outliers) > 0:
    print("\n异常值年份:")
    display(outliers[['timestamp', 'load_mw']])

### ⚠️ 关键决策: 为什么**不删除**这些异常值？

在电力领域，统计上的"异常值"往往是**真实的极端事件**：
- 2008年金融危机后的用电量骤降
- 2020年疫情封锁期间的负荷坍塌
- 极端天气导致的空调全开

**删除这些数据 = 删除最重要的信号。**

这就是 PITFALLS.md 中的 "spike-as-noise" 反模式:
> 别把尖峰当噪声——尖峰就是信号。

XGBoost 和其他树模型天然善于处理极值（不像线性回归那样被异常值拖偏），
所以保留这些极值 = 保留模型在极端场景下的泛化能力。

IQR 检测在这里的作用是**告警**而非**清除**——
让你知道数据中有哪些时期是异常的，
但把这些异常留给模型自己去学习。

---
## 步骤 3: 执行清洗管道

In [ ]:
df_clean = clean_data(df_raw)

# 验证 Schema
result = validate_schema(df_clean)
if result['valid']:
    print("✅ Schema 验证通过")
else:
    print("❌ Schema 验证失败:")
    for issue in result['issues']:
        print(f"  - {issue}")

print(f"\n清洗前后对比:")
print(f"  行数: {len(df_raw)} → {len(df_clean)}")
print(f"  缺失值: {df_raw['load_mw'].isna().sum()} → {df_clean['load_mw'].isna().sum()}")
print(f"  时区: {df_raw['timestamp'].dtype} → {df_clean['timestamp'].dtype}")

## 📝 学习笔记

- IQR 方法是检测异常值的统计工具，不要用来删除数据
- 电力数据中，尖峰 = 信号，不是噪声
- 前向填充 (ffill) 是处理时序缺失值的最简单有效方法
- 干净的 Schema 是下游模块的"数据合约"

**下一步: Notebook 03 → 特征工程**

### 思考题

1. IQR 检测到异常值后为什么不删除？什么情况下删除异常值反而是错误的？
2. 前向填充和后向填充有什么区别？为什么电力数据先 ffill 再 bfill？
3. 如果数据是北京时间(UTC+8)，清洗后变成 UTC，预测结果的时间怎么理解？

